# Lecture 2.4 — Hands-On: Build a File Explorer Agent

In this notebook you will build a **File Explorer Agent** — an agent that autonomously explores a project directory, reads every file, and produces a clean summary report.

You describe the goal in plain English. The agent handles the rest.

### What we build
| Part | What we do |
|------|------------|
| 1 | Set up MODEL_NAME and create a realistic sample project programmatically |
| 2 | Verify the project — print the directory structure and every file's contents |
| 3 | Build the File Explorer Agent using `Glob` and `Read` tools |
| 4 | Change the prompt — discover your primary control lever |

> **This notebook is self-contained.** Run every cell from top to bottom.

In [ ]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

In [ ]:
# Load the API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

The `MODEL_NAME` variable below controls which Claude model every agent in this notebook uses.

For this course we use **`claude-haiku-4-5`** — it is fast and cost-efficient, which is a good choice while you are learning the SDK.

You are free to change this to any model you have access to:
- `"claude-sonnet-4-5"` — more capable, higher cost
- `"claude-opus-4-5"` — most capable, highest cost

Every `ClaudeAgentOptions` call in this notebook passes `model=MODEL_NAME`, so you only need to change it here.

In [ ]:
# Model configuration
# Change this to use a different Claude model
# e.g. "claude-sonnet-4-6", "claude-opus-4-7"
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## The Colab Filesystem

Before we create any files, it helps to understand where those files actually live.

**Colab runs on a Linux virtual machine (Ubuntu).** Every Colab session starts with a completely fresh filesystem.

- **`/content/`** is your working directory — think of it as the Colab equivalent of your home folder
- **Files only last for the session** — when the runtime disconnects or resets, everything under `/content/` is wiped
- **This is why we always create sample files programmatically** — every run starts clean and predictable, no matter what happened in a previous session
- **Agent tools work on this filesystem** — `Read`, `Glob`, `Grep` all operate on `/content/` exactly as they would on a real machine
- **You can ignore `/content/sample_data/`** — Colab creates this with some CSV files by default; our agent will not touch it

## Part 1 — Creating the Sample Project

### Why we create files programmatically

Three reasons:

1. **Full control** — We decide exactly what files exist and what they contain
2. **Reproducibility** — Every student who runs this notebook gets identical files
3. **Predictability** — Because the files are fixed, we know what the agent's summary should say

The project is a small but realistic Python library: a README, an authentication module, a utilities module, and a test file.

In [ ]:
# Create the sample project structure and files
import os

os.makedirs("/content/sample_project/src", exist_ok=True)
os.makedirs("/content/sample_project/tests", exist_ok=True)

with open("/content/sample_project/README.md", "w") as f:
    f.write("# Sample Project\n")
    f.write("This is a demo project for the File Explorer Agent.\n")
    f.write("It contains an authentication module, utility functions, and tests.\n")

with open("/content/sample_project/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write('    """Authenticate a user with username and password."""\n')
    f.write("    return user == 'admin' and password == 'secret'\n\n")
    f.write("def logout(user):\n")
    f.write('    """Log out the specified user."""\n')
    f.write("    return True\n\n")
    f.write("def reset_password(user, new_password):\n")
    f.write('    """Reset the password for the specified user."""\n')
    f.write("    return True\n")

with open("/content/sample_project/src/utils.py", "w") as f:
    f.write("# Utility functions\n\n")
    f.write("def format_date(date):\n")
    f.write('    """Format a date object as YYYY-MM-DD string."""\n')
    f.write("    return date.strftime('%Y-%m-%d')\n\n")
    f.write("def format_currency(amount):\n")
    f.write('    """Format a number as a currency string."""\n')
    f.write("    return f'${amount:.2f}'\n\n")
    f.write("def validate_email(email):\n")
    f.write('    """Check if an email address is valid."""\n')
    f.write("    return '@' in email\n")

with open("/content/sample_project/tests/test_auth.py", "w") as f:
    f.write("# Tests for auth module\n\n")
    f.write("def test_login():\n")
    f.write('    """Test the login function."""\n')
    f.write("    assert login('admin', 'secret') == True\n\n")
    f.write("def test_logout():\n")
    f.write('    """Test the logout function."""\n')
    f.write("    assert logout('admin') == True\n")

print("Sample project created successfully.")

## Part 2 — Verifying What Was Created

Before we run the agent, let us print out exactly what we just created — the directory structure and every file's contents.

This is good practice: we know what the agent is going to read, so we can verify its summary against what we see here. No black boxes.

In [ ]:
# Show the directory structure and all file contents
import os

# Directory structure
print("Project structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/sample_project"):
    level = root.replace("/content/sample_project", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

# File contents
files_to_show = [
    "/content/sample_project/README.md",
    "/content/sample_project/src/auth.py",
    "/content/sample_project/src/utils.py",
    "/content/sample_project/tests/test_auth.py",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

## Part 3 — Building the File Explorer Agent

We now have a project on disk with four files. We are going to build an agent that explores it autonomously.

**Two tools:**
- **`Glob`** — finds files by pattern, like a powerful `find` command (e.g. `**/*.py` finds all Python files)
- **`Read`** — opens and reads any file in the working directory

**The pattern:** We give the agent a goal in plain English using a numbered prompt. It figures out the tool sequence itself — you do not tell it to use `Glob` before `Read`. It infers the right order from the goal.

We extract the final result using `hasattr(message, "result")` — the `ResultMessage` pattern from Lecture 2.3.

In [ ]:
# The File Explorer Agent
# Uses Glob + Read to autonomously explore the project and produce a summary report

#Type code below

## What Just Happened?

Notice what the agent did — and what **you** did not do.

- You did not tell it to use `Glob` before `Read`
- You did not write a loop to open each file
- You described a **goal** — the agent figured out the tool sequence itself

That is the **agent loop** from Lecture 1.4 running live on a real task:

1. Receive the goal (your prompt)
2. Decide which tool to call first
3. Call the tool, get the result
4. Use the result to decide the next step
5. Repeat until the goal is met
6. Return the final answer as a `ResultMessage`

You provided the goal. The agent handled everything else.

---

**Expected output** (your exact wording will vary — Claude writes these fresh each time):

```
Project Summary — /content/sample_project
==========================================

README.md
  A project description explaining this is a demo project containing
  authentication, utility, and test modules.

src/auth.py
  An authentication module with login, logout, and reset_password
  functions for managing user access.

src/utils.py
  A utility module with functions for formatting dates, currency values,
  and validating email addresses.

tests/test_auth.py
  A test module containing tests for the login and logout functions
  in the authentication module.
```

## Part 4 — Experimenting with the Prompt

### The prompt is your primary control lever

The prompt is the most powerful tool you have when building agents. **Change the prompt, change the agent** — without touching a single line of code.

Below we use the **same tools** and the **same code structure**, but swap the prompt. Instead of asking for file summaries, we ask for a function inventory.

Same agent. Different goal. Different output.

In [ ]:
# Modified agent: find all Python files and list function names
# Same tools (Glob + Read), same ResultMessage pattern —
# but a different prompt produces a completely different result.

#Type code below

## Summary — What We Built and What We Learned

### What we built

A **File Explorer Agent** that:
- Autonomously discovers all files in a project using `Glob`
- Reads and understands every file using `Read`
- Produces a structured summary report
- Can be redirected to a completely different task by changing only the prompt

### Key things we established

| Concept | What it is |
|---------|------------|
| `MODEL_NAME` | One variable to control which Claude model every agent uses |
| Colab filesystem | Fresh Ubuntu VM every session — `/content/` as home, files created programmatically |
| Verification pattern | Print the project before the agent runs — no surprises |
| `Glob` + `Read` | Two tools that together give complete filesystem read access |
| `hasattr(message, "result")` | The clean `ResultMessage` pattern for extracting the final answer |
| Prompt as control lever | One prompt change → completely different agent behaviour |

### Expected output from Cell 15 (approximate)

```
Python Functions by File
========================

src/auth.py
  - login(user, password)
  - logout(user)
  - reset_password(user, new_password)

src/utils.py
  - format_date(date)
  - format_currency(amount)
  - validate_email(email)

tests/test_auth.py
  - test_login()
  - test_logout()
```

---

### What's next

**Lecture 2.5 — Troubleshooting Common Setup Errors**  
If anything in this notebook did not run cleanly, Lecture 2.5 covers the most common setup problems and how to fix them.

---
*Great work. You built your first complete agent.*